<a href="https://colab.research.google.com/github/gabrielxbox/Formacao_Completa_Intelig-ncia_Artificial/blob/main/DeteccaoObjetos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

In [2]:
MODEL_CONFIG = "MobileNetSSD_deploy.prototxt" # pesso do modelo
MODEL_WEIGHTS = "MobileNetSSD_deploy.caffemodel" # configuração
CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat","bottle","bus","car","cat",
           "chair","cow","dinigtable","dog","horse","motorbike","person","pottedplant","sheep",
           "sofa","train","tvmonitor"]

In [ ]:
net = cv2.dnn.readNetFromCaffe(MODEL_CONFIG, MODEL_WEIGHTS)

In [4]:
def detect_person_in_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    person_detected_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
          print("fim do video ou erro de leitura")
          break
        frame_count += 1
        (h, w) = frame.shape[:2]
        blob = cv2.dnn.blobFromImage(frame, 0.007843, (300, 300), 127.5)
        net.setInput(blob)
        detections = net.forward()

        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > 0.9:
                idx = int(detections[0, 0, i, 1])
                if CLASSES[idx] == "person":
                    box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                    (startX, startY, endX, endY) = box.astype("int")

                    label = f"{CLASSES[idx]}: {confidence * 100:.2f}"
                    cv2.rectangle(frame, (startX, startY), (endX, endY),(0,255,0),2)
                    cv2.putText(frame, label, (startX, startY - 10),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    print(f"pessoa detectada no freme {frame_count}")
                    cv2_imshow(frame)

                    detected_from_path = f"pessoa_frame+{frame_count}.jpg"
                    cv2.imwrite(detected_from_path)
                    print(f"freme salvo com {detected_from_path}")

                    cap.release()
                    return "Pessoa detectada no video"
    cap.release()
    return "Nenhuma Pessoa detectada no video"